# Stenosis Arcade — Val Set Visualization
# labels (bounding boxes) + labels_old (segmentation polygons)

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.collections import PatchCollection
from PIL import Image

BASE = "/home/dsa/stenosis/data/stenosis_arcade/val"
IMG_DIR = os.path.join(BASE, "images")
LBL_DIR = os.path.join(BASE, "labels")       # bounding boxes (YOLO)
LBL_OLD_DIR = os.path.join(BASE, "labels_old")  # segmentation polygons

# Collect first 100 images (sorted numerically)
img_paths = sorted(glob.glob(os.path.join(IMG_DIR, "*.png")),
                   key=lambda p: int(os.path.splitext(os.path.basename(p))[0]))
img_paths = img_paths[:100]
print(f"Found {len(img_paths)} images to visualize")

In [ ]:
def parse_yolo_bbox(label_path, img_w, img_h):
    """Parse YOLO bbox labels: class cx cy w h -> list of (class_id, x1, y1, x2, y2)"""
    bboxes = []
    if not os.path.exists(label_path):
        return bboxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls = int(parts[0])
            cx, cy, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            x1 = (cx - w/2) * img_w
            y1 = (cy - h/2) * img_h
            x2 = (cx + w/2) * img_w
            y2 = (cy + h/2) * img_h
            bboxes.append((cls, x1, y1, x2, y2))
    return bboxes


def parse_yolo_polygon(label_path, img_w, img_h):
    """Parse YOLO polygon labels: class x1 y1 x2 y2 ... -> list of (class_id, polygon_xy)"""
    polys = []
    if not os.path.exists(label_path):
        return polys
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 7:
                continue
            cls = int(parts[0])
            coords = list(map(float, parts[1:]))
            # coords = x1,y1,x2,y2,...  (normalized)
            xs = [coords[i] * img_w for i in range(0, len(coords), 2)]
            ys = [coords[i] * img_h for i in range(1, len(coords), 2)]
            poly_xy = list(zip(xs, ys))
            polys.append((cls, poly_xy))
    return polys


BBOX_COLORS = {0: "lime", 1: "cyan"}       # class 0, class 1
POLY_COLOR = "red"

print("Helper functions defined.")

In [ ]:
COLS = 5
ROWS = 20  # 100 images => 20 rows x 5 cols
fig, axes = plt.subplots(ROWS, COLS, figsize=(COLS * 4, ROWS * 4))
axes = axes.flatten()

for idx, img_path in enumerate(img_paths):
    ax = axes[idx]
    stem = os.path.splitext(os.path.basename(img_path))[0]

    # Load image
    img = Image.open(img_path)
    img_w, img_h = img.size
    ax.imshow(img, cmap="gray")
    ax.set_title(stem, fontsize=9)
    ax.axis("off")

    # --- Draw segmentation polygons (labels_old) in red ---
    lbl_old_path = os.path.join(LBL_OLD_DIR, f"{stem}.txt")
    polys = parse_yolo_polygon(lbl_old_path, img_w, img_h)
    for cls, poly_xy in polys:
        polygon = plt.Polygon(poly_xy, closed=True, fill=True,
                              facecolor="red", edgecolor="red",
                              alpha=0.25, linewidth=1.5)
        ax.add_patch(polygon)

    # --- Draw bounding boxes (labels) ---
    lbl_path = os.path.join(LBL_DIR, f"{stem}.txt")
    bboxes = parse_yolo_bbox(lbl_path, img_w, img_h)
    for cls, x1, y1, x2, y2 in bboxes:
        color = BBOX_COLORS.get(cls, "yellow")
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                  linewidth=2, edgecolor=color,
                                  facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1 - 2, f"cls {cls}", color=color, fontsize=7,
                fontweight="bold", va="bottom")

plt.tight_layout()
plt.show()
print("Done — 100 val images with labels (green/cyan bbox) + labels_old (red polygon)")